In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ranksums
import json

In [ ]:
expts = {'XLI2':'Schisto',
         'XCE1':'CoV',
         'JYH3':'BoNT/A'}
expt_order = ['XLI2','JYH3','XCE1']
alg_order = ['individual','meta','scoper','mmseqs']

scores = []
for e in expt_order:
    for a in alg_order:
        sc = pd.read_csv(f'{e}_{a}_scores_extra.csv',index_col=0)
        sc['expt'] = e
        sc['alg'] = a

        if a == 'scoper':
            sc['max_dist'] = 1 - sc['max_dist'] #fix mistake
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score']/50
        elif a=='mmseqs':
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score_v3']/(-50)
        elif a=='individual':
            sc['max_dist'] = sc['cut']
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score']/50
        else:
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score']/50
        
        scores.append(sc)

scores = pd.concat(scores,axis=0).reset_index(drop=True)
scores

In [ ]:
stability = scores.groupby(['alg','name','expt'])['ari_to_full'].min().reset_index().rename({'ari_to_full':'stability'},axis=1)
scores = scores.merge(stability,on=['alg','name','expt'],how='left')
scores

In [ ]:
score_order = ['silhouette','scaled_ranksum_score_v3','ari','stability']

summary = []
best_settings = {}
for alg,group in scores[scores['factor']==1].groupby('alg'):
    for expt in expt_order:
        other_expts = [e for e in expt_order if e != expt]
        best_name = group.pivot(index='name',columns='expt',values=score_order)[:][other_expts].prod(axis=1).idxmax()
        best_settings[(alg,expt)] = best_name
        
        best_scores = []
        for sc in score_order:
            best_scores.append(group.loc[(group['name']==best_name)&(group['expt']==expt),sc].values[0]) #score for overall best params
        summary.append([alg, expts[expt]]+best_scores)

summary = pd.DataFrame(summary,columns=['alg','expt']+score_order)
summary

In [ ]:
colorblind = sns.color_palette("colorblind")
tab10 = plt.get_cmap('tab10')
letters = 'ABCD'

name_convert = {
    'ari':'ARI',
    'silhouette':'Silhouette',
    'scaled_ranksum_score_v3':'Phenotypic Quality',
    'stability':'Stability'
}

color_id = {
    'individual':9,
    'meta':0,
    'scoper':1,
    'mmseqs':2
}

alg_name = {
    'individual':'NanoMAP (ind)',
    'meta':'NanoMAP (meta)',
    'scoper':'SCOPer',
    'mmseqs':'MMseqs2'
}

summary['alg_name'] = summary['alg'].map(lambda alg: alg_name[alg])

plt.figure(figsize=[12,8])
for i,sc in enumerate(score_order):
    ax = plt.subplot(2,2,i+1)
    sns.barplot(data=summary, x='expt', y=sc, hue='alg_name',
                hue_order=[alg_name[alg] for alg in alg_order], legend=(i==1), palette=[colorblind[color_id[x]] for x in alg_order])

    for l, label in enumerate(ax.get_xticklabels()):
        label.set_color(tab10(l+3))

    if sc=='silhouette':
        plt.yticks([0,0.1,0.2,0.3,0.4])
    
    if i==1:
        plt.legend(loc=[0.25,0.63],fontsize=10)

    plt.xlabel('')

    plt.ylabel(name_convert[sc],fontsize=16)

    plt.xticks(fontsize=16,fontweight='bold')
    plt.yticks(fontsize=16)

    ax.text(-0.15, 1.05, letters[i], fontsize=32, transform=ax.transAxes)
plt.subplots_adjust(hspace=0.35,wspace=0.35)
plt.savefig('fig3.png',dpi=300,bbox_inches='tight')

In [ ]:
best_settings

In [ ]:
json.dump({'-'.join(k):best_settings[k] for k in best_settings},open('best_params.json','w'))